## setup

In [ ]:
# import stuff
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# load data from folder
folder = Path('../data')

# filtered dataset
listings = pd.read_csv(folder / 'listings.csv', low_memory = False)

# unfiltered dataset, needed for eda 
raw_listings = pd.read_csv(folder / 'unfiltered_listings.csv', low_memory = False)

## EDA

### dataset understanding
- check columns, data types, shape, head
- check property categories
- validate completeness

In [ ]:
# ensuring datasets loaded in correctly (unfiltered)
print(raw_listings.shape)
raw_listings.head()

In [ ]:
# ensuring datasets loaded in correctly (filtered)
print(listings.shape)
listings.head()

In [ ]:
listings.columns

In [ ]:
listings.dtypes

### missing value analysis
- null-count summary table
- flag columns with >90% missing values
- which columns to drop or retain?

In [ ]:
# null-ct summary table
listings_null_ct = listings.isnull().sum()
listings_null_pct = listings.isnull().mean() * 100

listings_null_summary = pd.DataFrame({
    'null ct': listings_null_ct,
    'null pct': listings_null_pct
})

listings_null_summary.sort_values(by = 'null pct', ascending = False).head(10)

In [ ]:
# save null-ct summary table to csv
listings_null_summary.to_csv(folder / 'listings_null_summary.csv', index = True)

In [ ]:
# find cols w >90% nulls
over_90 = listings_null_summary[listings_null_summary['null pct'] > 90]
print('Columns with over 90% nulls:', over_90.shape[0])
# output: 13

# prepare list of cols to flag if they have >90% nulls
flag_over_90 = over_90.index.tolist()
flag_over_90

In [ ]:
# what if we adjusted the threshold to >50% nulls
over_50 = listings_null_summary[listings_null_summary['null pct'] > 50]
print('Columns with over 50% nulls:', over_50.shape[0])
# output: 32

flag_over_50 = over_50.index.tolist()
flag_over_50

In [ ]:
# exclude core fields from flagged list if there are any
core_fields = ['ClosePrice', 'ListPrice', 'OriginalListPrice',
               'LivingArea', 'LotSizeAcres', 'BedroomsTotal',
               'BathroomsTotalInteger', 'DaysOnMarket', 'YearBuilt']

for field in core_fields:
    if field in flag_over_50:
        flag_over_50.remove(field)

# check if any flagged cols were removed
print(len(flag_over_50))
# 31

### numeric distribution summary
- ``.describe()``
- data visualizations (histograms, boxplots, percentile summaries)
- identify extreme outliers

In [ ]:
numeric_fields = ['ClosePrice', 'ListPrice', 'OriginalListPrice', 
                  'LivingArea', 'LotSizeAcres', 'BedroomsTotal', 
                  'BathroomsTotalInteger', 'DaysOnMarket', 'YearBuilt']

listings_summary = listings[numeric_fields].describe()
listings_summary

In [ ]:
# visualizations for each numeric field (histograms, boxplots, and percentile summaries, and identify extreme outliers)
for col in ['ClosePrice', 'LivingArea', 'DaysOnMarket']:
    plt.figure(figsize=(12, 4))
    
    # histogram
    plt.subplot(1, 3, 1)
    plt.hist(listings[col].dropna())
    plt.title(f'{col} Histogram')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    
    # boxplot
    plt.subplot(1, 3, 2)
    plt.boxplot(listings[col].dropna())
    plt.title(f'{col} Boxplot')
    plt.ylabel(col)
    
    # percentile summary
    plt.subplot(1, 3, 3)
    percentiles = [25, 50, 75]
    vals = listings[col].quantile([p/100 for p in percentiles])
    plt.bar([f'{p}%' for p in percentiles], vals)
    plt.title(f'{col} Percentile Summary')
    plt.ylabel(col)
    
    plt.tight_layout()
    plt.show()

In [ ]:
outliers = []

# find outliers
for field in numeric_fields:
    # focus on one column at a time, convert to numeric and drop nulls
    col = pd.to_numeric(listings[field], errors = 'coerce').dropna()

    # find lower, upper bounds and IQR
    q1 = col.quantile(0.25)
    q3 = col.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    low_outliers = int((col < lower_bound).sum())
    high_outliers = int((col > upper_bound).sum())

    outliers.append({
        'field': field,
        'min': col.min(),
        'max': col.max(),
        'lower_bound': lower_bound,
        'upper_bound': upper_bound,
        'low_outlier_ct': low_outliers,
        'high_outlier_ct': high_outliers,
        'total_outlier_ct': low_outliers + high_outliers,
        'outlier_pct': round((low_outliers + high_outliers) / len(col) * 100, 2)
    })

In [ ]:
outlier_summary = pd.DataFrame(outliers)
outlier_summary

In [ ]:
# visualize without outliers
for col in ['ClosePrice', 'LivingArea', 'DaysOnMarket']:
    plt.figure(figsize=(12, 4))
    
    # filter out outliers
    q1 = listings[col].quantile(0.25)
    q3 = listings[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    filtered_col = listings[(listings[col] >= lower_bound) & (listings[col] <= upper_bound)][col]
    
    # histogram
    plt.subplot(1, 3, 1)
    plt.hist(filtered_col.dropna())
    plt.title(f'{col} Histogram (No Outliers)')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    
    # boxplot
    plt.subplot(1, 3, 2)
    plt.boxplot(filtered_col.dropna())
    plt.title(f'{col} Boxplot (No Outliers)')
    plt.ylabel(col)

    # percentile summary
    plt.subplot(1, 3, 3)
    percentiles = [25, 50, 75]
    vals = filtered_col.quantile([p/100 for p in percentiles])
    plt.bar([f'{p}%' for p in percentiles], vals)
    plt.title(f'{col} Percentile Summary (No Outliers)')
    plt.ylabel(col)
    
    plt.tight_layout()
    plt.show()

### save filtered dataset

In [ ]:
# save filtered dataset as new csv
# listings.to_csv('../data/listings.csv', index = False)

## EDA questions

### What is the Residential vs other property type share?

In [ ]:
property_pct = round(raw_listings['PropertyType'].value_counts(normalize = True) * 100, 2)
property_pct

The table above is from the raw listings dataset which shows that about 63.62% of property types are Residential and the remaining 36.38% are non-residential.

### What are the median & average close prices?

In [ ]:
print('Median close price:', round(listings['ClosePrice'].median(), 2))
print('Average close price:', round(listings['ClosePrice'].mean(), 2))

For the sold dataset, the median close price (860,000) is a lower value than the average (1,203,607), meaning that this column follows a right-skewed distribution. This is supported by the visualization generated earlier, without outliers, where there is a cluster of values on the left side and there is a right tail for higher values.

### Look at days on market distribution

In [ ]:
listings['DaysOnMarket'].describe()

From the summary table for ``DaysOnMarket``, we can see that the average number of days a listing has been on the market is around 21 days whereas the median number of days is 11. Likewise to ``ClosePrice``, this column follows a right-skewed distribution as the median value is lower than the average, and is also supported by the visualization generated above. Besides the mean and median values, we also need to be aware of the minimum value which goes into the negatives. When performing data cleaning, we will need to convert invalid values (negatives, nulls) into zeroes.

### What percentage of homes sold above vs below list price?

In [ ]:
above_list_price_pct = round((listings[listings['ClosePrice'] > listings['ListPrice']].shape[0] / listings.shape[0]) * 100, 2)
print(above_list_price_pct)
# output: 12.74

below_list_price_pct = round((listings[listings['ClosePrice'] < listings['ListPrice']].shape[0] / listings.shape[0]) * 100, 2)
print(below_list_price_pct)
# output: 10.83

at_list_price_pct = round((listings[listings['ClosePrice'] == listings['ListPrice']].shape[0] / listings.shape[0]) * 100, 2)
print(at_list_price_pct)
# output: 5.36

About 12.74% of homes sold above list price and about 10.83% homes sold below list price, but homes are slightly more likely to be sold above the listed price by about 2%. Additionally, about 5.36% of homes are sold at the listed price.

### Are there any apparent date consistency issues (e.g close date before listing date)?

In [ ]:
# convert date columns to datetime
close_date = pd.to_datetime(listings['CloseDate'], errors = 'coerce')
list_date = pd.to_datetime(listings['ListingContractDate'], errors = 'coerce')
bought_date = pd.to_datetime(listings['PurchaseContractDate'], errors = 'coerce')

# close date before listing date
close_before_listing = int((close_date < list_date).sum())
print('Close date before listing date:', close_before_listing)

# close date before bought date
close_before_bought = int((close_date < bought_date).sum())
print('Close date before purchase date:', close_before_bought)

With June's data added to the dataset, we have 85 listings where the close date is before the listing date, and 266 listings where the close date is before the purchase date, which does not make sense. We'll have to be mindful of these rows when performing data cleaning.

### Which counties have the highest median prices?

In [ ]:
# see all unique counties
print(len(listings['CountyOrParish'].unique()))
listings['CountyOrParish'].unique()

In [ ]:
# find median close prices per county
listings_median_prices = listings.groupby('CountyOrParish')['ClosePrice'].median()
listings_median_prices = listings_median_prices.sort_values(ascending = False).reset_index(name = 'median_ClosePrice')
listings_median_prices.head(10)

The table above shows the top 10 counties with the highest median close prices, with the first county being San Mateo with $1,760,000 and the tenth highest county being San Diego with $900,000. About half of the counties in this list are either near northern California or the Bay Area and the rest are near southern California.